# SATMM Santander: alerta temprana de movimientos en masa

Reto avanzado, Hackathon UDES. Documentación en `docs/`.

In [ ]:
import sys
sys.path.insert(0, 'src')

import pandas as pd
from satmm import datos, boletin

pd.set_option('display.max_columns', 30)

## 1. Carga y limpieza

Todos los cruces se hacen por `codigo_dane`, nunca por nombre.

In [ ]:
panel = datos.cargar_panel()
emergencias = datos.cargar_emergencias()
municipios = datos.cargar_municipios()
panel.shape, emergencias.shape, municipios.shape

In [ ]:
datos.diagnostico(panel, emergencias, municipios)

**Problemas encontrados:** ver la tabla. Los datos llegan limpios (sin nulos, duplicados ni nombres inconsistentes). Hallazgos que sí afectan el modelo:

- `lluvia_mm` y `eventos_mes` son del mismo mes: **no se conocen el día 1** → se excluyen (fuga).
- `eventos_12m` de 2015 incluye historia de 2014 que no está en el panel → 2015 solo se usa para rezagos.
- Desbalance: ~8 % de meses con movimiento en masa.

In [ ]:
emergencias['evento'].value_counts()

In [ ]:
panel.groupby('mes')['hubo_mm'].mean().round(3).rename('tasa_hubo_mm').to_frame().T

## 2. Variables sin fuga

Cada fila del mes *t* usa solo información hasta *t − 1*. Las climatologías (`mm_hist_mes`, `lluvia_anom`) usan **solo años anteriores** al de la fila, así que no dependen del pliegue ni filtran el futuro. 2015 se usa solo para alimentar rezagos.

In [ ]:
from satmm import variables

tabla = variables.construir(panel, emergencias, municipios)
print(tabla.shape, '| excluidas por fuga:', variables.EXCLUIDAS)
tabla[['codigo_dane', 'anio', 'mes'] + variables.FEATURES + ['hubo_mm']].head()

In [ ]:
# Evidencia de la fuga: la lluvia del mismo mes 'parece' el doble de útil que la del mes anterior
tabla[['lluvia_mm', 'lluvia_t1', 'lluvia_2m', 'mm_12m', 'mm_hist_mes', 'mes_sin2', 'hubo_mm']].corr()['hubo_mm'].round(3)

## 3. Validación temporal y líneas base

Ventana expansiva por años: entrena con 2016…(a−1) y valida con el año *a* completo, para a = 2021…2024. **2025 no se toca** hasta la prueba final.

- **A:** repetir el mismo mes del año anterior.
- **B:** climatología del municipio en ese mes (solo años previos).

In [ ]:
from satmm import validacion as va

[(p.anio_valida, f'{p.anios_train[0]}–{p.anios_train[-1]}') for p in va.pliegues()]

In [ ]:
modelos = {
    'A: mismo mes año anterior': (None, va.linea_base_anio_anterior, 0.5),
    'B: climatología': (None, va.linea_base_climatologia, 0.1),
    'Logística': (va.logistica, None, 0.5),
}
res = va.evaluar(tabla, modelos)
va.resumen(res)

In [ ]:
res.pivot(index='anio', columns='modelo', values='pr_auc').round(3)

### Evidencia de la fuga
Misma logística, pero agregando `lluvia_mm` (lluvia del mismo mes, desconocida el día 1). La mejora es **falsa**: en producción ese dato no existe.

In [ ]:
feats_fuga = variables.FEATURES + ['lluvia_mm']
fuga = va.evaluar(tabla, {'Logística + lluvia_mm (FUGA)': (va.logistica, None, 0.5)}, features=feats_fuga)
pd.concat([res[res.modelo == 'Logística'], fuga]).groupby('modelo')[['pr_auc', 'recall', 'precision']].mean().round(3)

## 4. Modelo, umbral y semáforo

### Selección de modelo
XGBoost **sin peso de clase**: mejor PR-AUC y probabilidades calibradas (Brier ≈ climatología o mejor), porque el boletín publica la probabilidad. El desbalance se maneja con el umbral, no con pesos.

In [ ]:
from satmm import modelo as mo

comparacion = va.evaluar(tabla, {
    'A: mismo mes año anterior': (None, va.linea_base_anio_anterior, 0.5),
    'B: climatología': (None, va.linea_base_climatologia, 0.1),
    'Logística': (va.logistica, None, 0.5),
    'XGBoost': (mo.xgboost, None, 0.1),
})
comparacion.pivot(index='anio', columns='modelo', values='pr_auc').round(3)

In [ ]:
comparacion.groupby('modelo')[['pr_auc', 'brier', 'top10_precision']].mean().round(3).sort_values('pr_auc', ascending=False)

### Umbral (elegido en 2024 y congelado)
- **Amarillo:** probabilidad ≥ umbral que detecta el 70 % de los meses con movimiento en masa en 2024 → vigilancia.
- **Rojo:** los **10 municipios más altos de cada mes** que además superan el amarillo → capacidad del consejo para movilizar recursos.

In [ ]:
F = variables.FEATURES
tr24, va24 = va.separar(tabla, va.pliegues()[-1])
p24 = mo.xgboost().fit(tr24[F], tr24['hubo_mm']).predict_proba(va24[F])[:, 1]
umbrales = mo.elegir_umbrales(va24, p24, recall_amarillo=0.70, rojos_mes=10)
umbrales

### Prueba final: 2025 (se evalúa una sola vez)

In [ ]:
final = tabla[tabla['anio'] <= 2024]
prueba = tabla[tabla['anio'] == 2025].copy()
modelo_final = mo.xgboost().fit(final[F], final['hubo_mm'])
prueba['p'] = modelo_final.predict_proba(prueba[F])[:, 1]
prueba['nivel'] = mo.semaforo(prueba, prueba['p'].to_numpy(), umbrales)

pd.DataFrame({
    'XGBoost': va.metricas(prueba, prueba['p'].to_numpy(), umbrales.amarillo),
    'A: mismo mes año anterior': va.metricas(prueba, va.linea_base_anio_anterior(prueba), 0.5),
    'B: climatología': va.metricas(prueba, va.linea_base_climatologia(prueba), 0.1),
}).round(3)

In [ ]:
pd.Series(mo.evaluar_semaforo(prueba, prueba['p'].to_numpy(), umbrales)).round(3)

In [ ]:
# Semáforo mes a mes en 2025 frente a lo que pasó
prueba.groupby('mes').apply(lambda g: pd.Series({
    'rojos': (g.nivel == 'Rojo').sum(), 'amarillos': (g.nivel == 'Amarillo').sum(),
    'eventos_reales': g.hubo_mm.sum(), 'detectados': g[g.nivel != 'Verde'].hubo_mm.sum()})).T

### Falsos negativos
Municipios con movimiento en masa en 2025 que el semáforo dejó en verde. Patrones: la mayoría ocurre en **meses secos**, cuando el sistema casi no alerta, y la mitad **no tenía movimientos en masa en los 12 meses previos**. El modelo depende de la temporada y del historial; un municipio que no reporta es casi invisible (subregistro).

In [ ]:
fn = prueba[(prueba.hubo_mm == 1) & (prueba.nivel == 'Verde')]
print(f'{len(fn)} de {int(prueba.hubo_mm.sum())} meses con evento quedaron en verde')
fn[['municipio', 'mes', 'p', 'lluvia_2m', 'mm_12m', 'mm_hist_mes']].sort_values('p')

### Alertas del mes para el mapa y el boletín

In [ ]:
from satmm import explicacion as ex

ANIO, MES = 2025, 4  # abril: temporada de lluvias
alertas = mo.alertas_mes(modelo_final, tabla, ANIO, MES, umbrales, municipios)
alertas = ex.agregar_razones(alertas, modelo_final, F)
alertas['nivel'].value_counts()

In [ ]:
alertas[['ranking', 'municipio', 'nivel', 'p', 'lluvia_2m', 'mm_12m', 'hubo_mm']].head(12)

## 5. Explicación (SHAP)

SHAP reparte cada predicción entre las variables. Las 14 variables técnicas se agrupan en **7 factores** que entiende un consejo municipal (lluvia reciente, temporada, relieve, historial…).

In [ ]:
import shap, matplotlib.pyplot as plt

sv = ex.valores_shap(modelo_final, final[F])
shap.summary_plot(sv.to_numpy(), final[F], show=False, max_display=14)
plt.title('Aporte de cada variable (2016–2024)'); plt.tight_layout(); plt.savefig('out/shap_global.png', dpi=120); plt.show(); plt.close('all')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ex.por_factor(sv).abs().mean().sort_values().plot.barh(ax=ax, title='Peso promedio de cada factor (|SHAP|)')
fig.tight_layout(); fig.savefig('out/shap_factores.png', dpi=120); plt.show()

### Chequeo de sentido común
El riesgo debe **subir** con la lluvia, el historial y la altitud. Correlación de Spearman entre el valor de la variable y su aporte SHAP (positivo = sube el riesgo):

In [ ]:
pd.Series({c: ex.chequeo_monotonia(modelo_final, final[F], c) for c in ['lluvia_t1', 'lluvia_2m', 'mm_12m', 'altitud_m', 'mm_hist_mes']}).round(3)

In [ ]:
shap.dependence_plot('lluvia_2m', sv.to_numpy(), final[F], interaction_index=None, show=False)
plt.title('Dónde empieza a subir el riesgo por lluvia'); plt.tight_layout(); plt.show(); plt.close('all')

### Explicación por municipio
Para cada municipio se toman los 3 factores **propios** que más suben su probabilidad (la temporada se reporta aparte porque es igual para todos los municipios del mes).

In [ ]:
for _, a in alertas[alertas.nivel == 'Rojo'].head(5).iterrows():
    print(f"{a.ranking:>2}. {a.municipio} ({a.nivel}, {100*a.p:.1f} %)")
    for r in a.razones:
        print('     •', ex.texto_razon(r, MES, boletin.num))

In [ ]:
# Cascada del municipio #1: de la tasa base a su probabilidad
i = alertas.index[0]
exp = shap.TreeExplainer(modelo_final)(alertas[F])
shap.plots.waterfall(exp[i], max_display=10, show=False)
plt.title(alertas.loc[i, 'municipio']); plt.tight_layout(); plt.show()

## 6. Mapa de alertas

In [ ]:
from satmm import mapa

m = mapa.construir(alertas, f'Alertas {MES:02d}/{ANIO} (provisional)')
m.save('out/mapa_alertas.html')
m

## 7. Boletín verificado

In [ ]:
texto, _ = boletin.generar(alertas.iloc[0], emergencias, len(alertas))
open('out/boletin.md', 'w').write(texto)
from IPython.display import Markdown
Markdown(texto)

### Prueba del verificador
Se agrega una cifra inventada al boletín: el verificador debe señalarla.

In [ ]:
c = boletin.cifras(alertas.iloc[0], emergencias, len(alertas))
boletin.verificar(texto + '\nSe estiman 350 viviendas en riesgo.', c)

## Tablero
Un mapa por mes de 2025, boletines verificados para **todos** los municipios en rojo, y el tablero `out/tablero.html` que lo junta (abrir en el navegador).

In [ ]:
from satmm import tablero

m25 = va.metricas(prueba, prueba['p'].to_numpy(), umbrales.amarillo)
sem = mo.evaluar_semaforo(prueba, prueba['p'].to_numpy(), umbrales)
metricas_2025 = {
    'recall_alerta': sem['recall_rojo_o_amarillo'], 'precision_rojo': sem['precision_rojo'],
    'tasa_base': m25['tasa_real'], 'pr_auc': m25['pr_auc'],
    'pr_auc_A': va.metricas(prueba, va.linea_base_anio_anterior(prueba), 0.5)['pr_auc'],
    'pr_auc_B': va.metricas(prueba, va.linea_base_climatologia(prueba), 0.1)['pr_auc'],
    'recall_A': va.metricas(prueba, va.linea_base_anio_anterior(prueba), 0.5)['recall'],
    'fn': len(fn), 'eventos': int(prueba['hubo_mm'].sum()),
}
tablero.generar(modelo_final, tabla, umbrales, municipios, emergencias, metricas_2025)

## 8. Qué no puede anticipar el sistema